# 🚦 Road Accident Severity Prediction
## Exploratory Data Analysis & Machine Learning

This notebook covers:
1. Dataset loading and preview
2. Data cleaning and preprocessing
3. Exploratory Data Analysis (EDA)
4. Data visualization
5. Model training and evaluation

## 📦 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.style.use('dark_background')
sns.set_palette('husl')

print('✅ Libraries imported successfully')

## 📂 2. Load Dataset

In [ ]:
df = pd.read_csv('dataset/accident_data.csv')
print(f'Dataset shape: {df.shape}')
df.head(10)

In [ ]:
# Data types inspection
print('\n📋 Column Data Types:')
print(df.dtypes)
print('\n📊 Statistical Summary:')
df.describe()

## 🧹 3. Handle Missing Values

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nTotal missing: {df.isnull().sum().sum()}')

# Visualize
fig, ax = plt.subplots(figsize=(10, 4))
null_pct = (df.isnull().sum() / len(df) * 100)
null_pct[null_pct > 0].plot(kind='bar', ax=ax, color='#e74c3c')
ax.set_title('Missing Value Percentage by Column')
ax.set_ylabel('Missing %')
plt.tight_layout()
plt.show()

In [ ]:
# Fill missing values
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].fillna(df[col].mode()[0])
df['Driver_Age'] = df['Driver_Age'].fillna(df['Driver_Age'].median())

print(f'Missing values after cleaning: {df.isnull().sum().sum()}')

## 📊 4. Accident Severity Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
vc = df['Accident_Severity'].value_counts()
colors = {'Minor': '#27ae60', 'Serious': '#f39c12', 'Fatal': '#e74c3c'}
bar_colors = [colors.get(k, '#95a5a6') for k in vc.index]
axes[0].bar(vc.index, vc.values, color=bar_colors, edgecolor='white')
axes[0].set_title('Accident Severity Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
for i, (idx, val) in enumerate(vc.items()):
    axes[0].text(i, val + 10, str(val), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(vc.values, labels=vc.index, colors=bar_colors,
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Severity Proportion', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 🌦️ 5. Weather Conditions Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
weather_sev = df.groupby(['Weather_Conditions', 'Accident_Severity']).size().unstack(fill_value=0)
weather_sev.plot(kind='bar', ax=ax, color=['#e74c3c', '#27ae60', '#f39c12'],
                 edgecolor='white', width=0.8)
ax.set_title('Accident Count by Weather Condition & Severity', fontsize=14, fontweight='bold')
ax.set_xlabel('Weather Condition')
ax.set_ylabel('Number of Accidents')
ax.legend(title='Severity')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 🛣️ 6. Road Type Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
road_sev = df.groupby(['Road_Type', 'Accident_Severity']).size().unstack(fill_value=0)
road_sev.plot(kind='bar', stacked=True, ax=ax,
              color=['#e74c3c', '#27ae60', '#f39c12'], edgecolor='white', width=0.7)
ax.set_title('Accident Severity by Road Type (Stacked)', fontsize=14, fontweight='bold')
ax.set_xlabel('Road Type')
ax.set_ylabel('Number of Accidents')
ax.legend(title='Severity')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 🔥 7. Correlation Heatmap

In [ ]:
from sklearn.preprocessing import LabelEncoder

df_enc = df.copy()
for col in ['Weather_Conditions', 'Road_Type', 'Light_Conditions', 'Time_of_Day', 'Accident_Severity']:
    le = LabelEncoder()
    df_enc[col] = le.fit_transform(df_enc[col])

plt.figure(figsize=(10, 7))
corr = df_enc.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, linewidths=0.5, square=True,
            cbar_kws={'label': 'Correlation Coefficient'})
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 🤖 8. Model Training & Evaluation

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
import joblib

FEATURES = ['Weather_Conditions', 'Road_Type', 'Speed_Limit',
            'Light_Conditions', 'Time_of_Day', 'Number_of_Vehicles', 'Driver_Age']
TARGET = 'Accident_Severity'

X = df_enc[FEATURES]
y = df_enc[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

rf = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print(f'\nAccuracy: {accuracy_score(y_test, y_pred)*100:.2f}%')
print('\nClassification Report:')
print(classification_report(y_test, y_pred))

In [ ]:
# Feature Importance Plot
importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()
fig, ax = plt.subplots(figsize=(10, 5))
importances.plot(kind='barh', ax=ax, color='#e74c3c', edgecolor='white')
ax.set_title('Feature Importance — Random Forest', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('\n✅ Analysis complete! Run train_model.py to save the model.')